# fastapriori — Feature Demo

Complete walkthrough of all package features: pairwise associations, metrics, triplets, utilities, and backend selection.

In [18]:
import pandas as pd
import numpy as np
import fastapriori
from fastapriori import (
    find_associations,
    find_triplets,
    get_top_associations,
    filter_associations,
    to_heatmap,
    to_graph,
)

print(f"fastapriori v{fastapriori.__version__}")

fastapriori v0.1.0


In [29]:
df = pd.read_csv("10m.csv")

In [30]:
df.shape

(9987027, 2)

## 1. Sample Dataset

A small grocery store dataset to illustrate all features clearly.

In [19]:
# 10 transactions, 6 products
transactions = {
    1: ["Bread", "Butter", "Milk"],
    2: ["Bread", "Butter"],
    3: ["Bread", "Milk", "Eggs"],
    4: ["Butter", "Milk", "Eggs"],
    5: ["Bread", "Butter", "Milk", "Eggs"],
    6: ["Bread", "Cheese"],
    7: ["Milk", "Cheese"],
    8: ["Bread", "Butter", "Cheese"],
    9: ["Bread", "Milk"],
    10: ["Butter", "Eggs"],
}

rows = [(txn, item) for txn, items in transactions.items() for item in items]
df = pd.DataFrame(rows, columns=["Transaction ID", "Item"])

print(f"Rows: {len(df)} | Transactions: {df['Transaction ID'].nunique()} | Items: {df['Item'].nunique()}")
df.head(10)

Rows: 26 | Transactions: 10 | Items: 5


,Transaction ID,Item
0,1,Bread
1,1,Butter
2,1,Milk
3,2,Bread
4,2,Butter
5,3,Bread
6,3,Milk
7,3,Eggs
8,4,Butter
9,4,Milk


## 2. Pairwise Associations (Pandas Backend)

`find_associations()` computes all pairwise co-occurrences with 10 metrics: support, confidence, lift, conviction, leverage, cosine, jaccard.

In [20]:
# All associations with no filtering
result = find_associations(
    df,
    transaction_col="Transaction ID",
    item_col="Item",
    min_support=None,
    min_confidence=None,
)
print(f"Total association rules: {len(result)}")
result

Total association rules: 18


,item_A,item_B,instances,support,confidence,lift,conviction,leverage,cosine,jaccard
0,Bread,Butter,4,0.4,0.571429,0.952382,0.9333,-0.02,0.6172,0.4444
1,Bread,Milk,4,0.4,0.571429,0.952382,0.9333,-0.02,0.6172,0.4444
2,Bread,Eggs,2,0.2,0.285714,0.714285,0.8400,-0.08,0.3780,0.2222
3,Bread,Cheese,2,0.2,0.285714,0.952380,0.9800,-0.01,0.4364,0.2500
4,Butter,Bread,4,0.4,0.666667,0.952381,0.9000,-0.02,0.6172,0.4444
5,Butter,Milk,3,0.3,0.500000,0.833333,0.8000,-0.06,0.5000,0.3333
6,Butter,Eggs,3,0.3,0.500000,1.250000,1.2000,0.06,0.6124,0.4286
7,Butter,Cheese,1,0.1,0.166667,0.555557,0.8400,-0.08,0.2357,0.1250
8,Cheese,Bread,2,0.2,0.666667,0.952381,0.9000,-0.01,0.4364,0.2500
9,Cheese,Butter,1,0.1,0.333333,0.555555,0.6000,-0.08,0.2357,0.1250


In [21]:
# With thresholds: min_support=0.1, min_confidence=0.3
filtered = find_associations(
    df,
    transaction_col="Transaction ID",
    item_col="Item",
    min_support=0.1,
    min_confidence=0.3,
)
print(f"Filtered rules: {len(filtered)}")
filtered

Filtered rules: 14


,item_A,item_B,instances,support,confidence,lift,conviction,leverage,cosine,jaccard
0,Bread,Butter,4,0.4,0.571429,0.952382,0.9333,-0.02,0.6172,0.4444
1,Bread,Milk,4,0.4,0.571429,0.952382,0.9333,-0.02,0.6172,0.4444
2,Butter,Bread,4,0.4,0.666667,0.952381,0.9000,-0.02,0.6172,0.4444
3,Butter,Milk,3,0.3,0.500000,0.833333,0.8000,-0.06,0.5000,0.3333
4,Butter,Eggs,3,0.3,0.500000,1.250000,1.2000,0.06,0.6124,0.4286
5,Cheese,Bread,2,0.2,0.666667,0.952381,0.9000,-0.01,0.4364,0.2500
6,Cheese,Butter,1,0.1,0.333333,0.555555,0.6000,-0.08,0.2357,0.1250
7,Cheese,Milk,1,0.1,0.333333,0.555555,0.6000,-0.08,0.2357,0.1250
8,Eggs,Butter,3,0.3,0.750000,1.250000,1.6000,0.06,0.6124,0.4286
9,Eggs,Bread,2,0.2,0.500000,0.714286,0.6000,-0.08,0.3780,0.2222


## 3. Polars Backend

Same API, same results — just pass `backend="polars"`. Uses a self-join approach under the hood.

In [22]:
result_polars = find_associations(
    df,
    transaction_col="Transaction ID",
    item_col="Item",
    min_support=0.1,
    min_confidence=0.3,
    backend="polars",
)
print(f"Polars backend rules: {len(result_polars)}")
result_polars

Polars backend rules: 14


,item_A,item_B,instances,support,confidence,lift,conviction,leverage,cosine,jaccard
0,Cheese,Bread,2,0.2,0.666667,0.952381,0.9000,-0.01,0.4364,0.2500
1,Butter,Eggs,3,0.3,0.500000,1.250000,1.2000,0.06,0.6124,0.4286
2,Bread,Milk,4,0.4,0.571429,0.952382,0.9333,-0.02,0.6172,0.4444
3,Bread,Butter,4,0.4,0.571429,0.952382,0.9333,-0.02,0.6172,0.4444
4,Milk,Butter,3,0.3,0.500000,0.833333,0.8000,-0.06,0.5000,0.3333
5,Eggs,Milk,3,0.3,0.750000,1.250000,1.6000,0.06,0.6124,0.4286
6,Cheese,Butter,1,0.1,0.333333,0.555555,0.6000,-0.08,0.2357,0.1250
7,Eggs,Bread,2,0.2,0.500000,0.714286,0.6000,-0.08,0.3780,0.2222
8,Butter,Bread,4,0.4,0.666667,0.952381,0.9000,-0.02,0.6172,0.4444
9,Milk,Bread,4,0.4,0.666667,0.952381,0.9000,-0.02,0.6172,0.4444


## 4. Triplets (3-Itemset Associations)

`find_triplets()` finds items that frequently appear together as groups of three. Each triplet (A,B,C) generates **3 directional rules**: A,B→C | A,C→B | B,C→A — each with different confidence and lift values.

Pass `frequent_pairs` from pairwise analysis to prune the search space (Apriori property).

In [24]:
# Find all triplets (no filtering)
triplets = find_triplets(
    df,
    transaction_col="Transaction ID",
    item_col="Item",
    min_support=None,
    min_confidence=None,
)
print(f"Triplet rules: {len(triplets)}")
triplets

Triplet rules: 5


,item_A,item_B,item_C,instances,support,confidence,lift
0,Bread,Butter,Milk,2,0.2,0.500000,0.833333
1,Bread,Eggs,Milk,2,0.2,1.000000,1.666667
2,Butter,Eggs,Milk,2,0.2,0.666667,1.111112
3,Bread,Butter,Eggs,1,0.1,0.250000,0.625000
4,Bread,Butter,Cheese,1,0.1,0.250000,0.833333


In [25]:
# Pruned triplets: only consider triplets where all 3 pairs are in filtered results
triplets_pruned = find_triplets(
    df,
    transaction_col="Transaction ID",
    item_col="Item",
    min_support=0.1,
    min_confidence=None,
    frequent_pairs=filtered,
)
print(f"Pruned triplet rules: {len(triplets_pruned)}")
triplets_pruned

Pruned triplet rules: 5


,item_A,item_B,item_C,instances,support,confidence,lift
0,Bread,Butter,Milk,2,0.2,0.500000,0.833333
1,Bread,Eggs,Milk,2,0.2,1.000000,1.666667
2,Butter,Eggs,Milk,2,0.2,0.666667,1.111112
3,Bread,Butter,Eggs,1,0.1,0.250000,0.625000
4,Bread,Butter,Cheese,1,0.1,0.250000,0.833333


## 5. Utility Functions

### 5a. Top Associations

Get the top N most associated items for a specific item, ranked by any metric.

In [26]:
# Top 5 items associated with "Bread", ranked by lift
top_bread = get_top_associations(result, "Bread", metric="lift", n=5)
top_bread

,item_A,item_B,instances,support,confidence,lift,conviction,leverage,cosine,jaccard
0,Bread,Butter,4,0.4,0.571429,0.952382,0.9333,-0.02,0.6172,0.4444
1,Bread,Milk,4,0.4,0.571429,0.952382,0.9333,-0.02,0.6172,0.4444
2,Butter,Bread,4,0.4,0.666667,0.952381,0.9000,-0.02,0.6172,0.4444
3,Cheese,Bread,2,0.2,0.666667,0.952381,0.9000,-0.01,0.4364,0.2500
4,Milk,Bread,4,0.4,0.666667,0.952381,0.9000,-0.02,0.6172,0.4444


In [27]:
# Top items where "Eggs" is the consequent (item_B) — "what predicts Eggs?"
top_eggs = get_top_associations(result, "Eggs", metric="confidence", n=5, role="consequent")
top_eggs

,item_A,item_B,instances,support,confidence,lift,conviction,leverage,cosine,jaccard
0,Butter,Eggs,3,0.3,0.500000,1.250000,1.20,0.06,0.6124,0.4286
1,Milk,Eggs,3,0.3,0.500000,1.250000,1.20,0.06,0.6124,0.4286
2,Bread,Eggs,2,0.2,0.285714,0.714285,0.84,-0.08,0.3780,0.2222


### 5b. Filter Associations

Filter the full result set by specific items.

In [28]:
# All rules involving Butter or Cheese (in either direction)
dairy_rules = filter_associations(result, ["Butter", "Cheese"], role="any")
dairy_rules

,item_A,item_B,instances,support,confidence,lift,conviction,leverage,cosine,jaccard
0,Bread,Butter,4,0.4,0.571429,0.952382,0.9333,-0.02,0.6172,0.4444
1,Bread,Cheese,2,0.2,0.285714,0.952380,0.9800,-0.01,0.4364,0.2500
2,Butter,Bread,4,0.4,0.666667,0.952381,0.9000,-0.02,0.6172,0.4444
3,Butter,Milk,3,0.3,0.500000,0.833333,0.8000,-0.06,0.5000,0.3333
4,Butter,Eggs,3,0.3,0.500000,1.250000,1.2000,0.06,0.6124,0.4286
5,Butter,Cheese,1,0.1,0.166667,0.555557,0.8400,-0.08,0.2357,0.1250
6,Cheese,Bread,2,0.2,0.666667,0.952381,0.9000,-0.01,0.4364,0.2500
7,Cheese,Butter,1,0.1,0.333333,0.555555,0.6000,-0.08,0.2357,0.1250
8,Cheese,Milk,1,0.1,0.333333,0.555555,0.6000,-0.08,0.2357,0.1250
9,Eggs,Butter,3,0.3,0.750000,1.250000,1.6000,0.06,0.6124,0.4286


### 5c. Heatmap

Pivot the results into an item x item matrix for heatmap visualization.

In [36]:
# Lift heatmap — values > 1 indicate positive association
heatmap = to_heatmap(result, metric="lift")
heatmap.style.background_gradient(cmap="RdYlGn", vmin=0.5, vmax=2.0).format("{:.2f}")

item_B,Bread,Butter,Cheese,Eggs,Milk
item_A,,,,,
Bread,0.00,0.95,0.95,0.71,0.95
Butter,0.95,0.00,0.56,1.25,0.83
Cheese,0.95,0.56,0.00,0.00,0.56
Eggs,0.71,1.25,0.00,0.00,1.25
Milk,0.95,0.83,0.56,1.25,0.00


In [37]:
# Confidence heatmap — P(B | A) for each pair
heatmap_conf = to_heatmap(result, metric="confidence")
heatmap_conf.style.background_gradient(cmap="Blues", vmin=0, vmax=1.0).format("{:.2f}")

item_B,Bread,Butter,Cheese,Eggs,Milk
item_A,,,,,
Bread,0.00,0.57,0.29,0.29,0.57
Butter,0.67,0.00,0.17,0.50,0.50
Cheese,0.67,0.33,0.00,0.00,0.33
Eggs,0.50,0.75,0.00,0.00,0.75
Milk,0.67,0.50,0.17,0.50,0.00


### 5d. Graph Export

Export associations as a NetworkX graph for network analysis or visualization.

In [30]:
import networkx as nx

# Export rules with lift >= 1.0 as a directed graph
G = to_graph(result, metric="lift", min_value=1.0)
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print("\nEdges (item_A -> item_B, lift):")
for u, v, d in G.edges(data=True):
    print(f"  {u} -> {v}  (lift={d['weight']:.4f}, confidence={d['confidence']:.4f})")

Nodes: 3
Edges: 4

Edges (item_A -> item_B, lift):
  Butter -> Eggs  (lift=1.2500, confidence=0.5000)
  Eggs -> Butter  (lift=1.2500, confidence=0.7500)
  Eggs -> Milk  (lift=1.2500, confidence=0.7500)
  Milk -> Eggs  (lift=1.2500, confidence=0.5000)


## 6. Larger Dataset — Skewed Distribution

Demonstrating on a more realistic dataset with Zipf-distributed item popularity.

In [15]:
import time

np.random.seed(42)
NUM_ITEMS, NUM_TXNS = 200, 50_000
items = [f"P-{i:04d}" for i in range(NUM_ITEMS)]
weights = 1.0 / np.arange(1, NUM_ITEMS + 1) ** 0.8
weights /= weights.sum()

rows = []
for t in range(NUM_TXNS):
    n = min(np.random.poisson(5) + 1, NUM_ITEMS)
    for item in np.random.choice(items, n, replace=False, p=weights):
        rows.append((t, item))

df_large = pd.DataFrame(rows, columns=["txn", "item"])
print(f"Rows: {len(df_large):,} | Transactions: {NUM_TXNS:,} | Items: {df_large['item'].nunique()}")

Rows: 300,524 | Transactions: 50,000 | Items: 200


In [31]:
t0 = time.perf_counter()
result_large = find_associations(
    df_large, "txn", "item",
    min_support=0.005, min_confidence=0.1,
    show_progress=True,
)
t1 = time.perf_counter()
print(f"\nTime: {t1 - t0:.2f}s | Rules: {len(result_large):,}")
result_large.sort_values("lift", ascending=False).head(15)

Counting co-occurrences: 100%|██████████████████████████████████████████████████████| 200/200 [00:00<00:00, 501.52it/s]


Time: 1.72s | Rules: 539


,item_A,item_B,instances,support,confidence,lift,conviction,leverage,cosine,jaccard
466,P-0097,P-0001,298,0.00596,0.353081,1.159620,1.0751,0.000820,0.0831,0.0189
385,P-0060,P-0003,265,0.00530,0.222315,1.156325,1.0386,0.000717,0.0783,0.0251
224,P-0025,P-0008,273,0.00546,0.120000,1.143511,1.0171,0.000685,0.0790,0.0377
329,P-0044,P-0002,395,0.00790,0.264214,1.133090,1.0422,0.000928,0.0946,0.0310
347,P-0049,P-0003,304,0.00608,0.215603,1.121414,1.0298,0.000658,0.0826,0.0284
463,P-0095,P-0000,419,0.00838,0.516646,1.108587,1.1047,0.000821,0.0964,0.0177
476,P-0104,P-0000,405,0.00810,0.512658,1.100030,1.0957,0.000737,0.0944,0.0171
420,P-0073,P-0002,273,0.00546,0.255618,1.096226,1.0301,0.000479,0.0774,0.0219
524,P-0151,P-0000,293,0.00586,0.509565,1.093393,1.0887,0.000501,0.0800,0.0124
477,P-0104,P-0001,262,0.00524,0.331646,1.089221,1.0406,0.000429,0.0755,0.0166


In [33]:
# Triplets on the larger dataset (pruned with pairwise results)
t0 = time.perf_counter()
triplets_large = find_triplets(
    df_large, "txn", "item",
    min_support=0.005, min_confidence=0.1,
    frequent_pairs=result_large,
    show_progress=True,
)
t1 = time.perf_counter()
print(f"\nTime: {t1 - t0:.2f}s | Triplet rules: {len(triplets_large):,}")
triplets_large.sort_values("lift", ascending=False).head(10)

Counting triplets: 100%|██████████████████████████████████████████████████████| 50000/50000 [00:00<00:00, 75747.74it/s]



Time: 2.43s | Triplet rules: 58


,item_A,item_B,item_C,instances,support,confidence,lift
41,P-0000,P-0008,P-0010,250,0.00500,0.100806,1.135970
36,P-0001,P-0004,P-0009,279,0.00558,0.112682,1.132938
56,P-0002,P-0003,P-0006,321,0.00642,0.142035,1.108609
45,P-0001,P-0002,P-0009,398,0.00796,0.109401,1.099950
52,P-0001,P-0002,P-0005,557,0.01114,0.153106,1.099110
13,P-0000,P-0002,P-0005,843,0.01686,0.151482,1.087452
49,P-0001,P-0005,P-0006,296,0.00592,0.138577,1.081619
17,P-0000,P-0001,P-0009,787,0.01574,0.107558,1.081420
39,P-0000,P-0008,P-0009,265,0.00530,0.106855,1.074351
32,P-0000,P-0007,P-0009,292,0.00584,0.106764,1.073437


## 7. Metric Interpretation Guide

| Metric | Range | What it means |
|--------|-------|---------------|
| **support** | [0, 1] | How common is this pair across all transactions |
| **confidence** | [0, 1] | Given item A, how likely is item B |
| **lift** | (0, +inf) | >1 = positive association, 1 = independent, <1 = negative |
| **conviction** | (0, +inf) | High = strong directional dependence |
| **leverage** | [-0.25, 0.25] | >0 = more co-occurrence than expected by chance |
| **cosine** | [0, 1] | Symmetric similarity between items |
| **jaccard** | [0, 1] | Overlap of transaction sets (intersection / union) |